In [1]:
from utils.analysis.portfolio import PortfolioAnalyzer, PortfolioConfig
from utils.data import DataManager
from utils.tools.config import ANALYSIS_START_DATE, ANALYSIS_END_DATE

In [2]:
# 📊 CONFIGURACIÓN DEL PORTFOLIO
# Valores por defecto vienen de config.py (PORTFOLIO_CONFIG)
# Personaliza aquí si necesitas valores diferentes:

config = PortfolioConfig(
    min_score=70.0,              # Puntaje mínimo de calidad (default: 60.0)
    max_companies=10,            # Máximo de empresas en portfolio (default: 10)
    max_per_sector=3,            # Máximo por sector (default: 3)
    selection_method='value', # Método: 'balanced', 'value', 'growth', 'total_score'
    weight_method='markowitz',       # Método: 'equal', 'score', 'markowitz'
    lookback_years=3             # Años históricos (default: 5)
)

print("✅ Configuración creada")
print(f"   Min Score: {config.min_score}")
print(f"   Max Companies: {config.max_companies}")
print(f"   Selection: {config.selection_method}")
print(f"   Weights: {config.weight_method}")
print(f"   Lookback: {config.lookback_years} años")

✅ Configuración creada
   Min Score: 70.0
   Max Companies: 10
   Selection: value
   Weights: markowitz
   Lookback: 3 años


In [3]:
# 🔧 INICIALIZACIÓN
# Crear instancia compartida de DataManager para eficiencia
data_manager = DataManager()

# Crear analyzer pasándole la configuración y el data_manager
analyzer = PortfolioAnalyzer(config=config, data_manager=data_manager)

print("\n📅 Fechas de análisis (desde config.py):")
print(f"   Inicio: {ANALYSIS_START_DATE}")
print(f"   Fin: {ANALYSIS_END_DATE}")
print("\n✅ Analyzer inicializado y listo para usar")


📅 Fechas de análisis (desde config.py):
   Inicio: 2020-01-01
   Fin: 2025-12-24

✅ Analyzer inicializado y listo para usar


In [4]:
# 📈 EJEMPLO 1: Analizar componentes de un índice
# Índices disponibles: 'SP500', 'NASDAQ100', 'DOW30'

print("🔍 Analizando S&P 500...")
result = analyzer.analyze_from_index('SP500')

if result.get('success'):
    print(f"\n✅ Portfolio creado exitosamente!")
    print(f"\n📊 Empresas seleccionadas: {len(result['tickers'])}")
    for ticker in result['tickers']:
        weight = result['weights'].get(ticker, 0)
        print(f"   {ticker}: {weight*100:.2f}%")
    
    print(f"\n📈 Métricas del portfolio:")
    metrics = result['metrics']
    for key, value in metrics.items():
        print(f"   {key}: {value}")
    
    print(f"\n📅 Período analizado:")
    print(f"   {result['period']['start']} → {result['period']['end']}")
else:
    print(f"❌ Error: {result.get('error')}")

🔍 Analizando S&P 500...
✅ Obtenidos 501 componentes del S&P 500
Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  5 of 5 completed


✅ Portfolio creado exitosamente!

📊 Empresas seleccionadas: 5
   SYF: 23.16%
   CF: 7.30%
   AFL: 23.34%
   NEM: 46.20%
   INCY: 0.00%

📈 Métricas del portfolio:
   total_score: 70.84175713840163
   sector_allocation: {'Financial Services': 0.46500932322295707, 'Basic Materials': 0.534990676777043, 'Healthcare': 8.94371110846219e-18}
   num_companies: 5

📅 Período analizado:
   2020-01-01 → 2025-12-24


In [5]:
# 📈 EJEMPLO 2: Analizar tickers específicos
# Útil para portfolios personalizados

custom_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'JPM', 'V', 'WMT']

print(f"🔍 Analizando {len(custom_tickers)} empresas personalizadas...")
result = analyzer.analyze(custom_tickers)

if result.get('success'):
    print(f"\n✅ Portfolio creado!")
    print(f"📊 Seleccionadas: {len(result['tickers'])}/{len(custom_tickers)}")
    print(f"\nDistribución de pesos:")
    for ticker in result['tickers']:
        weight = result['weights'].get(ticker, 0)
        score = result['analysis'][ticker]['scores']['total']
        print(f"   {ticker}: {weight*100:5.2f}% (Score: {score:.1f})")
else:
    print(f"❌ Error: {result.get('error')}")

🔍 Analizando 10 empresas personalizadas...
❌ Error: No hay empresas que cumplan criterios


In [6]:
# 📊 EJEMPLO 3: Comparar diferentes métodos de selección
# Métodos disponibles: 'balanced', 'value', 'growth', 'total_score'

test_tickers = ['AAPL', 'MSFT', 'GOOGL', 'META', 'NVDA', 'JPM', 'BAC', 'WMT', 'PG', 'JNJ']
methods = ['balanced', 'value', 'growth', 'total_score']

print("🔬 Comparando métodos de selección:\n")

for method in methods:
    config_test = PortfolioConfig(
        min_score=60.0,
        max_companies=5,
        selection_method=method,
        weight_method='equal'
    )
    
    analyzer_test = PortfolioAnalyzer(config=config_test, data_manager=data_manager)
    result = analyzer_test.analyze(test_tickers)
    
    if result.get('success'):
        print(f"📈 Método '{method}':")
        print(f"   Seleccionados: {', '.join(result['tickers'])}")
    else:
        print(f"❌ Método '{method}': {result.get('error')}")
    print()

🔬 Comparando métodos de selección:

📈 Método 'balanced':
   Seleccionados: NVDA, GOOGL, AAPL, BAC

📈 Método 'value':
   Seleccionados: BAC, GOOGL, NVDA, AAPL

📈 Método 'growth':
   Seleccionados: NVDA, AAPL, GOOGL, BAC

📈 Método 'total_score':
   Seleccionados: NVDA, GOOGL, BAC, AAPL



In [7]:
# 🎯 EJEMPLO 4: Portfolio optimizado con Markowitz
# Requiere datos históricos para calcular matriz de covarianza

print("🎯 Creando portfolio optimizado (Markowitz)...\n")

config_markowitz = PortfolioConfig(
    min_score=65.0,
    max_companies=8,
    selection_method='balanced',
    weight_method='markowitz',  # ← Optimización de Sharpe Ratio
    lookback_years=3
)

analyzer_markowitz = PortfolioAnalyzer(config=config_markowitz, data_manager=data_manager)
result = analyzer_markowitz.analyze(test_tickers)

if result.get('success'):
    print("✅ Portfolio Markowitz creado!")
    print("\n⚖️  Pesos optimizados:")
    for ticker, weight in sorted(result['weights'].items(), key=lambda x: x[1], reverse=True):
        print(f"   {ticker}: {weight*100:5.2f}%")
    
    print(f"\n📊 Total empresas: {len(result['tickers'])}")
else:
    print(f"❌ Error: {result.get('error')}")

🎯 Creando portfolio optimizado (Markowitz)...

Período: 2020-01-01 → 2025-12-24


[*********************100%***********************]  1 of 1 completed

✅ Portfolio Markowitz creado!

⚖️  Pesos optimizados:
   NVDA: 100.00%

📊 Total empresas: 1
